# Model External Validation

This document outlines the external validation procedures conducted for the RXNGraphormer model, utilizing established literature datasets to rigorously assess performance, robustness, and generalizability across diverse chemical reaction spaces. The validation framework leverages high-quality, peer-reviewed datasets spanning distinct reaction classes and experimental paradigms, enabling a comprehensive evaluation of the model’s predictive capabilities beyond the original training domain.

## Primary Validation Datasets

### Sulfoxonium Ylide Dataset
- **Reference**: [Lin et al., *Sci. China Chem.* **68**, 679–686 (2025)](https://doi.org/10.1007/s11426-024-2313-5)
- **Description**: This dataset comprises results from high-throughput experimentation on Ru-catalyzed P(O)O–H insertion reactions of sulfoxonium ylides. It features broad coverage of substrate scope and reaction conditions, with well-characterized α-phosphoryloxy ketone products and experimentally measured yields.

### Meta-C–H Functionalization Dataset
- **Reference**: [Selective functionalization of hindered meta-C–H bond of o-alkylaryl ketones, *Chem* (2022)](https://www.sciencedirect.com/science/article/pii/S2451929422004284)
- **Description**: A dataset derived from automated synthesis platforms performing meta-selective C–H functionalization. It includes sterically congested substrates and diverse coupling partners, presenting a challenging benchmark for selectivity prediction.

### Amide Coupling Reaction HTE Dataset
- **Reference**: [Zhang et al., *Sci. China Chem,* **16**, 11809-11822(2025)](https://pubs.rsc.org/en/content/articlehtml/2025/sc/d5sc03364k)
- **Description**: A preprint dataset of amide coupling reactions generated via machine-guided high-throughput experimentation (HTE). The dataset features high-quality yield measurements and was constructed using unbiased sampling strategies, ensuring broad chemical diversity and minimal selection bias.

### Amide Coupling Literature Dataset
- **Description**: A curated compilation of amide coupling reactions extracted from recent chemical literature (past three years). This dataset provides broad coverage of reagent systems, functional groups, and substrate architectures, serving as an independent benchmark for model generalizability.

### Non-USPTO ORDerly Dataset
- **Reference**: [ORDerly Dataset, *J. Chem. Inf. Model.*, DOI: 10.1021/acs.jcim.4c00292](https://doi.org/10.1021/acs.jcim.4c00292)
- **Description**: A non-USPTO chemical reaction dataset comprising carefully curated and standardized reactions from non-patent sources. The dataset is designed to mitigate biases inherent in USPTO-derived data and provides a more realistic distribution of chemical transformations. Both forward reaction prediction (reactants → products) and backward prediction (retrosynthesis; products → reactants) are evaluated on this dataset.

- **Subsets and Tasks**:

| Dataset Split                | Evaluation Task     | Pretrained Model(s) Used       |
|:----------------------------|:--------------------|:-------------------------------|
| `non_uspto_forward_mixed`   | Forward synthesis   | `USPTO-480k`, `USPTO-STEREO`   |
| `non_uspto_forward_separated` | Forward synthesis | `USPTO-STEREO`, `USPTO-STEREO` |
| `non_uspto_retro`           | Retrosynthesis      | `USPTO-full`                   |
| `non_uspto_retro`           | Retrosynthesis      | `USPTO-50K`                    |

---

## Methodological Limitations in Dataset Utilization

Due to the absence of mechanistic intermediate annotations in the source datasets, intermediate generation analysis—as reported in the original RXNGraphormer study—could not be implemented in this reproduction. To maintain consistency with experimental protocols and avoid potential artifacts from unverified intermediate predictions, all analyses were restricted to substrate–product pairs only. This limitation applies uniformly to datasets marked with **no mech**.

---

## Evaluation Protocol

For the Sulfoxonium Ylide and Meta-C–H Functionalization datasets (both marked **no mech**), model evaluation was performed using pre-trained optimal checkpoints. These models were initially trained and validated to convergence, with the best-performing weights saved based on validation set performance.

During evaluation, the saved optimal models were directly loaded, and inference was conducted on the test set by modifying only the data path in `model/parameters.json`. This approach ensures consistent and reproducible assessment without retraining or fine-tuning.

To support flexible and comprehensive evaluation, the `eval_regression_performance` function in `rxngraphormer/eval.py` was enhanced to accommodate four distinct evaluation configurations through the following binary switches:
- Use of a specific validation/test split (`specific_val`)
- Activation of intermediate information inference (`eval(pretrained_config.model.use_mid_inf)`)

This expanded framework enables all four logical combinations of these settings, allowing for systematic ablation studies and robust performance benchmarking across different modeling assumptions and data configurations.

## 1.Shell Example

In [19]:
#  python train_model.py --config_json ./config/Test/sulfoxonium_seed68909_parameters.json
#  python train_model.py --config_json ./config/Test/meta_C_H_parameters.json

## 2.Base Function

In [3]:
import glob
import os
import warnings

from rxngraphormer.eval import eval_regression_performance

warnings.filterwarnings("ignore")
cur_dir = os.getcwd()
father_dir = os.path.abspath(os.path.join(cur_dir, '..'))
all_results = {}

In [1]:
def evaluate_task(task_name, pattern, results_dict, father_dir, cur_dir, specific_val=True):
    original_dir = os.getcwd()
    try:
        os.chdir(father_dir)
        path_lst = sorted(glob.glob(pattern), key=lambda x: os.path.basename(x))
        all_r2, all_mae, all_preds, all_targets = [], [], [], []
        all_train_r2, all_train_mae = [], []

        for path in path_lst:
            name = os.path.basename(path)
            # tr_r2, tr_mae, _, _, r2, mae, preds, targets = eval_regression_performance(
            #     path, ckpt_file="valid_checkpoint.pt", scale=100,
            #     yield_constrain=True, specific_val=True, return_train_results=True
            # )
            r2, mae, preds, targets = eval_regression_performance(
                path, ckpt_file="valid_checkpoint.pt", scale=100,
                yield_constrain=True, specific_val=specific_val
            )
            # print( f"{task_name}_Train/{name}, Train R2: {tr_r2:.4f},Train MAE: {tr_mae:.4f},Test R2: {r2:.4f},Test MAE: {mae:.4f}")
            print(f"{name:<20}, R2: {r2:.10f}, MAE: {mae:.4f}")

            all_r2.append(r2)
            all_mae.append(mae)
            all_preds.append(preds)
            all_targets.append(targets)

        results_dict[task_name] = [all_r2, all_mae, all_preds, all_targets]
    finally:
        os.chdir(cur_dir)

## 3.Sulfoxonium Dataset

In [4]:
Sulfoxonium_tasks = {'Sulfoxonium': False, "Sulfoxonium_eval": True}
for task in Sulfoxonium_tasks.keys():
    evaluate_task(task, f"./model_path/Test/{task}", all_results, father_dir, cur_dir,
                  specific_val=Sulfoxonium_tasks[task])

Sulfoxonium         , R2: 0.9085094053, MAE: 5.7049
Sulfoxonium_eval    , R2: 0.6048046090, MAE: 9.4620


## 4.Meta_C_H Dataset（no mech）

In [5]:
Meta_C_H_tasks = {'Meta_C_H': False, "Meta_C_H_eval": True, "Meta_C_H_strict": False, "Meta_C_H_strict_eval": True}
for task in Meta_C_H_tasks.keys():
    evaluate_task(task, f"./model_path/Test/{task}", all_results, father_dir, cur_dir,
                  specific_val=Meta_C_H_tasks[task])

Meta_C_H            , R2: 0.8227147899, MAE: 5.7645
Meta_C_H_eval       , R2: 0.7758882090, MAE: 6.4859
Meta_C_H_strict     , R2: 0.8121455025, MAE: 5.9319
Meta_C_H_strict_eval, R2: -1.1941527247, MAE: 23.2152


## 5.Amide Coupling Reaction Datasets(HTE)
### 5.1 Introduced Intermediate

In [6]:
AmCouple_tasks = ['AmCouple_hte_random', 'AmCouple_hte_part', 'AmCouple_hte_full']
for task in AmCouple_tasks:
    evaluate_task(task, f"./model_path/Test/{task}", all_results, father_dir, cur_dir, specific_val=True)

AmCouple_hte_random , R2: 0.5761129783, MAE: 14.1392
AmCouple_hte_part   , R2: 0.5908010442, MAE: 14.3076
AmCouple_hte_full   , R2: 0.5763389855, MAE: 12.5638


### 5.2 No Intermediate Introduced

In [7]:
AmCouple_tasks = ['AmCouple_hte_random_nomech', 'AmCouple_hte_part_nomech', 'AmCouple_hte_full_nomech']
for task in AmCouple_tasks:
    evaluate_task(task, f"./model_path/Test/{task}", all_results, father_dir, cur_dir, specific_val=True)

AmCouple_hte_random_nomech, R2: 0.5843821814, MAE: 14.3055
AmCouple_hte_part_nomech, R2: 0.6582557780, MAE: 13.1205
AmCouple_hte_full_nomech, R2: 0.4430209522, MAE: 14.9298


### 5.3 Using Original Model's Intermediate

Selected condition Original model embedded with intermediate knowledge
- DCC Dataset
- EDC Dataset
- HATU Dataset
- PyBOP Dataset
- TBTU Dataset(no mech)
- HBTU Dataset(no mech)

In [8]:
AmCouple_conditions_tasks = [
    'DCC_random', 'DCC_part', 'DCC_full',
    'EDC_random', 'EDC_part', 'EDC_full',
    'HATU_random', 'HATU_part', 'HATU_full',
    'PyBOP_random', 'PyBOP_part', 'PyBOP_full',
    'HBTU_random_nomech', 'HBTU_part_nomech', 'HBTU_full_nomech',
    'TBTU_random_nomech', 'TBTU_part_nomech', 'TBTU_full_nomech',
]

for i, task in enumerate(AmCouple_conditions_tasks):
    evaluate_task(task, f"./model_path/Test/{task}", all_results, father_dir, cur_dir, specific_val=True)
    if (i + 1) % 3 == 0:
        print("-" * 50)

DCC_random          , R2: 0.3654025893, MAE: 16.6271
DCC_part            , R2: 0.2670526861, MAE: 15.9877
DCC_full            , R2: -0.4138258716, MAE: 13.0522
--------------------------------------------------
EDC_random          , R2: 0.2309699580, MAE: 18.4555
EDC_part            , R2: 0.1950737658, MAE: 18.3237
EDC_full            , R2: -0.0951265235, MAE: 22.2301
--------------------------------------------------
HATU_random         , R2: 0.0755858745, MAE: 19.0816
HATU_part           , R2: -0.0867442045, MAE: 20.5697
HATU_full           , R2: -0.3704953135, MAE: 16.3399
--------------------------------------------------
PyBOP_random        , R2: 0.3462648345, MAE: 15.5401
PyBOP_part          , R2: 0.3793608469, MAE: 14.0850
PyBOP_full          , R2: 0.2163931316, MAE: 15.5574
--------------------------------------------------
HBTU_random_nomech  , R2: 0.2276327815, MAE: 18.0151
HBTU_part_nomech    , R2: 0.0386743249, MAE: 18.7460
HBTU_full_nomech    , R2: -0.1052606158, MAE: 17.8

## 6.Amide Coupling Literature Dataset

In [9]:
 evaluate_task(task, f"./model_path/Test/AmCouple_lit", all_results, father_dir, cur_dir, specific_val=False)

AmCouple_lit        , R2: 0.3505837691, MAE: 12.4794


## 7. Non-USPTO ORDerly Dataset

In [1]:
 import os
import shutil
import random
import warnings
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger
from rxngraphormer.eval import reaction_prediction

warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')
cur_dir = os.getcwd()
father_dir = os.path.abspath(os.path.join(cur_dir, '..'))
os.chdir(father_dir)

def get_can_smi(smi, keep_stereo=True):
    """Minimal SMILES cleaning and canonicalization"""
    if not isinstance(smi, str):
        return None
    for token in [" ", "_EOS", "_PAD", "_UNK", "<eos>", "<pad>", "<unk>", "<s>", "</s>"]:
        smi = smi.replace(token, "")
    try:
        mols = [
            Chem.MolToSmiles(
                Chem.MolFromSmiles(f),
                isomericSmiles=keep_stereo
            )
            for f in smi.split('.') if f
        ]
        return ".".join(sorted(mols)) if mols else None
    except:
        return None


def run_evaluations(TASKS, sample_size=1000, seed=42):
    results = []

    for dataset, model, task_type in TASKS:
        print(f"\n[Running] Dataset: {dataset} | Model: {os.path.basename(model)}")

        src_path = f"./dataset/Test/Non_USPTO/{dataset}/src-test.txt"
        tgt_path = f"./dataset/Test/Non_USPTO/{dataset}/tgt-test.txt"

        src_lst = []
        with open(src_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip().replace(" ", "")
                if not line:
                    continue
                if dataset == "non_uspto_forward_separated":
                    line = line.replace('>', '.')
                    if line.endswith('.'):
                        line = line[:-1]
                src_lst.append(line)

        with open(tgt_path, 'r', encoding='utf-8') as f:
            tgt_lst = [line.strip().replace(" ", "") for line in f]

        if sample_size and sample_size < len(src_lst):
            random.seed(seed)
            idx = random.sample(range(len(src_lst)), sample_size)
            src_lst = [src_lst[i] for i in idx]
            tgt_lst = [tgt_lst[i] for i in idx]

        try:
            preds_df = reaction_prediction(model, src_lst, task_type=task_type)
            preds = preds_df.loc["Top-1"].tolist()
        except Exception as e:
            print(f"[Error] Inference failed: {e}")
            if os.path.exists("./rxn_emb_tmp"):
                shutil.rmtree("./rxn_emb_tmp", ignore_errors=True)
            continue

        inv_cnt, acc_sc, acc_wo_sc = 0, 0, 0

        for truth, pred in zip(tgt_lst, preds):
            t_sc = get_can_smi(truth, True)
            t_wo = get_can_smi(truth, False)
            p_sc = get_can_smi(pred, True)
            p_wo = get_can_smi(pred, False)

            if p_sc is None:
                inv_cnt += 1
            elif t_sc and p_sc == t_sc:
                acc_sc += 1

            if t_wo and p_wo and p_wo == t_wo:
                acc_wo_sc += 1

        total = len(src_lst)

        results.append({
            "dataset": dataset,
            "model": os.path.basename(model),
            "task": task_type,
            "invalid_rate (%)": inv_cnt / total * 100,
            "top1_acc_stereo (%)": acc_sc / total * 100,
            "top1_acc_no_stereo (%)": acc_wo_sc / total * 100,
        })

        if os.path.exists("./rxn_emb_tmp"):
            shutil.rmtree("./rxn_emb_tmp", ignore_errors=True)

    return pd.DataFrame(results)


TASKS = [
    ("non_uspto_forward_mixed",     "./model_path/USPTO_480k",   "forward-synthesis"),
    ("non_uspto_forward_mixed",     "./model_path/USPTO_STEREO", "forward-synthesis"),
    ("non_uspto_forward_separated", "./model_path/USPTO_480k",   "forward-synthesis"),
    ("non_uspto_forward_separated", "./model_path/USPTO_STEREO", "forward-synthesis"),
    ("non_uspto_retro",             "./model_path/USPTO_full",   "retro-synthesis"),
    ("non_uspto_retro",             "./model_path/USPTO_50k",    "retro-synthesis"),
]


final_results = run_evaluations(TASKS, sample_size=None)
final_results


[Running] Dataset: non_uspto_forward_mixed | Model: USPTO_480k


Using backend: pytorch


Loaded LocalMapper (version=202403) at device cpu
Model loaded successfully!


Processing...
100%|██████████| 50210/50210 [02:03<00:00, 408.12it/s]


[INFO] 50210 Saving...


...

Processing...
100%|██████████| 49206/49206 [01:29<00:00, 549.50it/s]


[INFO] 49206 Saving...


Done!


Done!


,dataset,model,task,invalid_rate (%),top1_acc_stereo (%),top1_acc_no_stereo (%)
0,non_uspto_forward_mixed,USPTO_480k,forward-synthesis,0.830512,83.337980,83.397730
1,non_uspto_forward_mixed,USPTO_STEREO,forward-synthesis,0.545708,83.343955,85.682135
2,non_uspto_forward_separated,USPTO_480k,forward-synthesis,0.794662,83.333997,83.393746
3,non_uspto_forward_separated,USPTO_STEREO,forward-synthesis,0.491934,83.258315,85.616411
4,non_uspto_retro,USPTO_full,retro-synthesis,4.186481,27.516969,27.945779
5,non_uspto_retro,USPTO_50k,retro-synthesis,3.046376,16.518311,16.640247


In [2]:
divider = "- " * 55
last_dataset = None

print("\n=== Non-USPTO Evaluation Results ===\n")

for _, row in final_results.iterrows():
    current_dataset = row["dataset"]

    if last_dataset is not None and current_dataset != last_dataset:
        print(divider)

    name_with_model = f"{current_dataset} ({row['model']})"

    metrics_str = (
        f"Invalid SMILES: {row['invalid_rate (%)']:.2f}% | "
        f"Accuracy (with SC): {row['top1_acc_stereo (%)']:.2f}% | "
        f"Accuracy (w/o SC): {row['top1_acc_no_stereo (%)']:.2f}%"
    )

    print(f"{name_with_model:<50} {row['task']:<20} {metrics_str}")

    last_dataset = current_dataset

if not final_results.empty:
    print(divider)


=== Non-USPTO Evaluation Results ===

non_uspto_forward_mixed (USPTO_480k)               forward-synthesis    Invalid SMILES: 0.83% | Accuracy (with SC): 83.34% | Accuracy (w/o SC): 83.40%
non_uspto_forward_mixed (USPTO_STEREO)             forward-synthesis    Invalid SMILES: 0.55% | Accuracy (with SC): 83.34% | Accuracy (w/o SC): 85.68%
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
non_uspto_forward_separated (USPTO_480k)           forward-synthesis    Invalid SMILES: 0.79% | Accuracy (with SC): 83.33% | Accuracy (w/o SC): 83.39%
non_uspto_forward_separated (USPTO_STEREO)         forward-synthesis    Invalid SMILES: 0.49% | Accuracy (with SC): 83.26% | Accuracy (w/o SC): 85.62%
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
non_uspto_retro (USPTO_full)                       retro-synthesis      Invalid SMILES: 4.19% | Accuracy (with SC): 27.52% | Accuracy (